In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import copy
import math

from torch.onnx.symbolic_opset9 import zero
from transformers import AutoTokenizer, AutoModel

## Muti-Header Attention Block

In [2]:
def clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

In [3]:
def attention(query, key, value, mask=None, dropout=None):
    "Compute 'Scaled Dot Product Attention'"
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    p_attn = scores.softmax(dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn

In [4]:
class MultiHeadsAttention(nn.Module):

    def __init__(self, d_model, h, dropout=0.1):
        super(MultiHeadsAttention, self).__init__()
        assert d_model % h == 0 # 断言投影矩阵维度可以整除头数
        self.d_model = d_model
        self.h = h
        self.d_k = d_model // h
        self.linears = clones(nn.Linear(d_model, d_model), 4)#QKVO四个线性投影变换
        self.attn = None
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, query, key, value, mask=None):
        if mask is not None:
            mask = mask.unsqueeze(1)
        nbatches = query.size(0)
        query, key, value = [
            lin(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2) #拆分多头
            for lin, x in zip(self.linears, (query, key, value))
        ]
        print(query.shape)
        print(key.shape  )
        print(value.shape)
        x, self.attn = attention(query, key, value, mask=mask, dropout=self.dropout)
        x = (
            x.transpose(1, 2)
            .contiguous()
            .view(nbatches, -1, self.h * self.d_k)
        )
        del query
        del key
        del value
        return self.linears[-1](x) #WO

In [7]:
def test_attention():
    torch.manual_seed(0)

    d_model = 512
    h = 64
    B = 2
    L = 10

    att = MultiHeadsAttention(d_model, h, dropout=0.0)  # dropout=0 方便复现
    att.eval()

    print("=" * 60)
    print(f"配置: d_model={d_model}, h={h}, d_k={d_model//h}")
    print(f"输入: B={B}, L={L}")
    print("=" * 60)

    # 输入必须是 [B, L, d_model]
    x = torch.randn(B, L, d_model)
    print(f"\n输入 x: {tuple(x.shape)}   # [B, L, d_model]")

    # 用自注意力：query = key = value = x
    out = att(x, x, x)
    print(f"输出 out: {tuple(out.shape)}   # [B, L, d_model]")

    assert out.shape == (B, L, d_model), "输出形状不对！"

    # 查看注意力矩阵
    print(f"\nattn: {tuple(att.attn.shape)}   # [B, h, L, L]")

    # 验证每行 softmax 和为 1
    row_sum = att.attn.sum(dim=-1)
    print(f"attn 每行和: min={row_sum.min():.4f}, max={row_sum.max():.4f}")
    assert torch.allclose(row_sum, torch.ones_like(row_sum), atol=1e-5)

In [8]:
if __name__ == "__main__":
    test_attention()

配置: d_model=512, h=64, d_k=8
输入: B=2, L=10

输入 x: (2, 10, 512)   # [B, L, d_model]
torch.Size([2, 64, 10, 8])
torch.Size([2, 64, 10, 8])
torch.Size([2, 64, 10, 8])
输出 out: (2, 10, 512)   # [B, L, d_model]

attn: (2, 64, 10, 10)   # [B, h, L, L]
attn 每行和: min=1.0000, max=1.0000


## 层归一化
    对每个特征维做归一化，然后做仿射变换，仿射变换让网络自行选择进行归一化的程度，否则任意的输入都会被归一化成正态分布

In [9]:
print(nn.Parameter(torch.ones(2, 3)))
print(nn.Parameter(torch.zeros(2, 3)))

Parameter containing:
tensor([[1., 1., 1.],
        [1., 1., 1.]], requires_grad=True)
Parameter containing:
tensor([[0., 0., 0.],
        [0., 0., 0.]], requires_grad=True)


In [ ]:
class LayerNorm(nn.Module):

    def __init__(self, features, eps=1e-6):
        super(LayerNorm, self).__init__()
        self.a_2 = nn.Parameter(torch.ones(features))
        self.b_2 = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2

## 残差连接

In [10]:
class SublayerConnection(nn.Module):

    def __init__(self, size, dropout):
        super(SublayerConnection, self).__init__()
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
        return x + self.dropout(sublayer(self.norm(x)))

## 位置编码

In [11]:
class PositionwiseFeedForward(nn.Module):

    def __init__(self, d_model, d_ff, dropout=0.1):
        super(PositionwiseFeedForward, self).__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.w_2(self.dropout(self.w_1(x).relu()))

## Encoder

## Code Test